# Recorded training experiments

This notebook starts **new experiments**. It does not resume the completed two-epoch baseline. Run initialization once, then deliberately run the training cell; expect roughly 16 minutes per training epoch plus evaluation. Do not use Run All casually. Use notebook 04 to inspect existing checkpoints without retraining. Shared logic lives in `voc_detection/`.

Saved outputs record the original experiments. Training cells now default to disabled; change the explicit flag only when intentionally starting or continuing training. The command-line workflow in README.md is the primary submission entry point.


In [2]:
from pathlib import Path
import sys

project_dir = next(
    path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (path / "voc_detection").is_dir() and (path / "configs").is_dir()
)
if str(project_dir) not in sys.path:
    sys.path.insert(0, str(project_dir))


## Initialize a fresh experiment
Rerunning this cell resets the model and optimizer.

In [3]:
import torch
import mlflow
from voc_detection.project import load_config, seed_everything, make_loader
from voc_detection.model import create_model
from voc_detection.engine import train_one_epoch
from voc_detection.evaluation import evaluate_model
from voc_detection.tracking import configure_tracking, log_evaluation

config = load_config()
seed_everything(config["seed"])
device = torch.device(config["device"])
training_dataset, training_loader = make_loader(config, "train", shuffle=True)
dev_dataset, dev_loader = make_loader(config, "dev", shuffle=False)
model = create_model(config["num_classes"]).to(device)
optimizer = torch.optim.SGD(
    [parameter for parameter in model.parameters() if parameter.requires_grad],
    lr=config["learning_rate"], momentum=config["momentum"], weight_decay=config["weight_decay"],
)
configure_tracking()
print("Fresh experiment initialized; no training has run.")


Fresh experiment initialized; no training has run.


## Train, evaluate, and save
This cell performs training. Each epoch gets its own checkpoint and fresh development predictions.

In [4]:
RUN_BASELINE_TRAINING = False  # Change deliberately to opt in to this training experiment.
if RUN_BASELINE_TRAINING:
    epochs = 1
    checkpoint_dir = project_dir / "mlflow_local" / "checkpoints"
    checkpoint_dir.mkdir(parents=True, exist_ok=True)
    with mlflow.start_run(run_name="mobilenetv3-clean-experiment") as run:
        mlflow.log_params({**config, "epochs": epochs})
        for split in ("train", "dev"):
            mlflow.log_artifact(str(project_dir / "splits" / f"{split}.txt"), artifact_path="splits")
        for epoch in range(1, epochs + 1):
            training_result = train_one_epoch(model, training_loader, optimizer, device, epoch)
            mlflow.log_metrics(training_result, step=epoch)
            checkpoint_path = checkpoint_dir / f"{run.info.run_id}_epoch_{epoch}.pt"
            torch.save({
                "epoch": epoch, "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(), "config": config,
                "mlflow_run_id": run.info.run_id,
                "torch_rng_state": torch.get_rng_state(),
                "loader_rng_state": training_loader.generator.get_state(),
            }, checkpoint_path)
            mlflow.log_artifact(str(checkpoint_path), artifact_path="checkpoints")
            evaluation = evaluate_model(model, dev_dataset, dev_loader, device)
            log_evaluation(evaluation, epoch)
            mlflow.set_tag("completed_epochs", epoch)
            print(f"Epoch {epoch}: development mAP@0.5 = {evaluation['map50']:.2%}")

else:
    print("Training disabled. Set RUN_BASELINE_TRAINING = True to run this experiment.")


Epoch 1 | Batch 1/2287 | Loss: 4.1271


KeyboardInterrupt: 

In [5]:
RUN_LR_EXPERIMENT = False  # Change deliberately to opt in to this training experiment.
if RUN_LR_EXPERIMENT:
    import torch
    import mlflow

    from voc_detection.project import load_config, seed_everything, make_loader
    from voc_detection.model import create_model
    from voc_detection.engine import train_one_epoch
    from voc_detection.evaluation import evaluate_model
    from voc_detection.tracking import configure_tracking, log_evaluation

    checkpoint_dir = project_dir / "mlflow_local" / "checkpoints"
    source_path = checkpoint_dir / "d43c960633a7477e8e8d719de3a27338_epoch_2.pt"

    checkpoint = torch.load(source_path, map_location="cpu", weights_only=True)

    config = {
        **load_config(),
        **checkpoint["config"],
        "learning_rate": 0.0001,
    }

    seed_everything(config["seed"])
    device = torch.device("cpu")

    training_dataset, training_loader = make_loader(config, "train", shuffle=True)
    dev_dataset, dev_loader = make_loader(config, "dev", shuffle=False)

    model = create_model(config["num_classes"], pretrained=True).to(device)
    model.load_state_dict(checkpoint["model_state_dict"])

    optimizer = torch.optim.SGD(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=config["learning_rate"],
        momentum=config["momentum"],
        weight_decay=config["weight_decay"],
    )

    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])

    for group in optimizer.param_groups:
        group["lr"] = config["learning_rate"]

    assert all(group["lr"] == 0.0001 for group in optimizer.param_groups)
    assert model.transform.min_size == (320,)

    configure_tracking()
    assert mlflow.active_run() is None

    epoch = checkpoint["epoch"] + 1

    with mlflow.start_run(run_name="epoch2-continuation-lr-0.0001") as run:
        mlflow.log_params({
            **config,
            "starting_epoch": checkpoint["epoch"],
            "additional_epochs": 1,
        })

        mlflow.set_tag("source_training_run", checkpoint["mlflow_run_id"])
        mlflow.set_tag("source_checkpoint", source_path.name)
        mlflow.set_tag("shuffle_policy", "Restarted from seed 42, matching epoch-3 control")

        for split in ("train", "dev"):
            mlflow.log_artifact(
                str(project_dir / "splits" / f"{split}.txt"),
                artifact_path="splits",
            )

        training_result = train_one_epoch(
            model, training_loader, optimizer, device, epoch
        )
        mlflow.log_metrics(training_result, step=epoch)

        output_path = checkpoint_dir / f"{run.info.run_id}_epoch_{epoch}.pt"

        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "mlflow_run_id": run.info.run_id,
            "torch_rng_state": torch.get_rng_state(),
            "loader_rng_state": training_loader.generator.get_state(),
        }, output_path)

        mlflow.log_artifact(str(output_path), artifact_path="checkpoints")

        evaluation = evaluate_model(model, dev_dataset, dev_loader, device)
        log_evaluation(evaluation, epoch)

        mlflow.set_tag("completed_epochs", epoch)

    print(f"Training loss: {training_result['train_loss']:.4f}")
    print(f"Development mAP@0.5: {evaluation['map50']:.2%}")
    print("Checkpoint:", output_path)
else:
    print("Training disabled. Set RUN_LR_EXPERIMENT = True to run this experiment.")


Epoch 3 | Batch 1/2287 | Loss: 1.3606
Epoch 3 | Batch 100/2287 | Loss: 0.5631
Epoch 3 | Batch 200/2287 | Loss: 0.5247
Epoch 3 | Batch 300/2287 | Loss: 0.9418
Epoch 3 | Batch 400/2287 | Loss: 0.6070
Epoch 3 | Batch 500/2287 | Loss: 0.3522
Epoch 3 | Batch 600/2287 | Loss: 0.5477
Epoch 3 | Batch 700/2287 | Loss: 0.1638
Epoch 3 | Batch 800/2287 | Loss: 0.2178
Epoch 3 | Batch 900/2287 | Loss: 0.7430
Epoch 3 | Batch 1000/2287 | Loss: 0.6561
Epoch 3 | Batch 1100/2287 | Loss: 0.8981
Epoch 3 | Batch 1200/2287 | Loss: 0.8345
Epoch 3 | Batch 1300/2287 | Loss: 0.4605
Epoch 3 | Batch 1400/2287 | Loss: 0.9420
Epoch 3 | Batch 1500/2287 | Loss: 0.6164
Epoch 3 | Batch 1600/2287 | Loss: 0.4657
Epoch 3 | Batch 1700/2287 | Loss: 0.6167
Epoch 3 | Batch 1800/2287 | Loss: 1.5114
Epoch 3 | Batch 1900/2287 | Loss: 0.9578
Epoch 3 | Batch 2000/2287 | Loss: 0.8396
Epoch 3 | Batch 2100/2287 | Loss: 0.5040
Epoch 3 | Batch 2200/2287 | Loss: 0.8590
Epoch 3 | Batch 2287/2287 | Loss: 0.7169
Evaluated 200/1144 images
Ev